# Salted Hashes

In [3]:
# The dump: username -> stored SHA-1 hash, plus the policy date for each account.
# (Pass_modified decides the password policy: before May 2010 = digits only.)
accounts = {
    "Sparky":   {"hash": "2834da08d58330d8dafbb2ac1c0f85f6b3b135ef", "modified": "2011-05-09"},
    "Mark123":  {"hash": "92e54f10103a3c511853c7098c04141f114719c1", "modified": "2010-06-03"},
    "superman": {"hash": "437fbc6892b38db6ac5bdbe2eab3f7bc924527d9", "modified": "2011-01-01"},
    "security": {"hash": "fafa4483874ec051989d53e1e432ba3a6c6b9143", "modified": "2010-01-11"},
    "Tomtom":   {"hash": "06f6fe0f73c6e197ee43eff4e5f7d10fb9e438b2", "modified": "2009-12-03"},
    "JillC":    {"hash": "f44f3b09df53c1c11273def13cacd8922a86d48c", "modified": "2010-12-20"},
}
for u, d in accounts.items():
    print(f"{u:10s} {d['hash']}  (set {d['modified']})")


Sparky     2834da08d58330d8dafbb2ac1c0f85f6b3b135ef  (set 2011-05-09)
Mark123    92e54f10103a3c511853c7098c04141f114719c1  (set 2010-06-03)
superman   437fbc6892b38db6ac5bdbe2eab3f7bc924527d9  (set 2011-01-01)
security   fafa4483874ec051989d53e1e432ba3a6c6b9143  (set 2010-01-11)
Tomtom     06f6fe0f73c6e197ee43eff4e5f7d10fb9e438b2  (set 2009-12-03)
JillC      f44f3b09df53c1c11273def13cacd8922a86d48c  (set 2010-12-20)


## Identify Hash Function

In [4]:
import hashlib

sample = accounts["Sparky"]["hash"]
print("stored hash:", sample)
print("length     :", len(sample), "hex chars =", len(sample)*4, "bits")

# Compare the lengths different algorithms produce for the SAME input:
for algo in ("md5", "sha1", "sha256"):
    h = hashlib.new(algo); h.update(b"hello")
    print(f"{algo:7s} -> {len(h.hexdigest())} hex chars")

# 40 hex chars / 160 bits == SHA-1. So CommonHash() is SHA-1.


stored hash: 2834da08d58330d8dafbb2ac1c0f85f6b3b135ef
length     : 40 hex chars = 160 bits
md5     -> 32 hex chars
sha1    -> 40 hex chars
sha256  -> 64 hex chars


## The construction (we know NO salt and NO password yet)

In [5]:
import itertools

def commonhash(salt, password):
    """Our hypothesis for the site's construction: SHA1(salt + password)."""
    return hashlib.sha1((salt + password).encode()).hexdigest()


## Finding the salt WITHOUT knowing any password

In [6]:
# Candidate salts = values visible in the lab document (the site domain + variants).
candidate_salts = ["exploringsecurity", "exploringsecurity.com",
                   "www.exploringsecurity.com", "ExploringSecurity"]

# Target the two PRE-2010 accounts (policy: digits-only, 5-7). We use their
# HASHES from the table -- NOT their passwords, which we do not have.
digit_targets = {accounts["security"]["hash"]: "security",
                 accounts["Tomtom"]["hash"]:   "Tomtom"}

def discover_salt(cand_salts):
    for salt in cand_salts:
        for length in range(5, 8):                 # 5-7 digits (from the policy clue)
            for combo in itertools.product("0123456789", repeat=length):
                pw = "".join(combo)
                hx = commonhash(salt, pw)
                if hx in digit_targets:
                    return salt, digit_targets[hx], pw   # salt + user + password together
        print(f"  rejected salt: {salt!r}  (unlocked nothing)")
    return None

salt, who, pw = discover_salt(candidate_salts)
SALT = salt                                        # lock in the DISCOVERED salt
print(f"\nDISCOVERED: salt={SALT!r}")
print(f"It unlocked account {who!r}, whose password turns out to be {pw!r}.")
print("We fed in no known password -- the brute force produced it.")


  rejected salt: 'exploringsecurity'  (unlocked nothing)
  rejected salt: 'exploringsecurity.com'  (unlocked nothing)

DISCOVERED: salt='www.exploringsecurity.com'
It unlocked account 'Tomtom', whose password turns out to be '54321'.
We fed in no known password -- the brute force produced it.


## Verification (sanity check)

In [7]:
stored = accounts[who]["hash"]
print("computed:", commonhash(SALT, pw))
print("stored  :", stored)
print("MATCH   :", commonhash(SALT, pw) == stored)   # True => construction + salt verified


computed: 06f6fe0f73c6e197ee43eff4e5f7d10fb9e438b2
stored  : 06f6fe0f73c6e197ee43eff4e5f7d10fb9e438b2
MATCH   : True


## Mask attack: spend what you KNOW before you spend compute

In [8]:
# Split accounts by the policy cutoff (May 2010) using Pass_modified.
pre_2010  = {u: d for u, d in accounts.items() if d["modified"] < "2010-05-01"}
post_2010 = {u: d for u, d in accounts.items() if d["modified"] >= "2010-05-01"}
print("digits-only (pre-2010):", list(pre_2010))
print("alphanumeric (post)   :", list(post_2010))

def mask_attack_digits(targets):
    want = {d["hash"]: u for u, d in targets.items()}
    found = {}
    for length in range(5, 8):
        for combo in itertools.product("0123456789", repeat=length):
            pw = "".join(combo)
            hx = commonhash(SALT, pw)
            if hx in want:
                found[want[hx]] = pw
                print(f"  [+] {want[hx]} -> {pw}")
        if len(found) == len(targets):
            break
    return found

import time
t = time.time()
print("Cracked:", mask_attack_digits(pre_2010), f"in {time.time()-t:.1f}s")


digits-only (pre-2010): ['security', 'Tomtom']
alphanumeric (post)   : ['Sparky', 'Mark123', 'superman', 'JillC']
  [+] Tomtom -> 54321
  [+] security -> 121298
Cracked: {'Tomtom': '54321', 'security': '121298'} in 0.4s


## Feasability Wall

In [9]:
def keyspace_report(charset_size, rate_per_sec=5_000_000):
    print(f"charset={charset_size}, assuming {rate_per_sec:,}/s on a CPU\n")
    for length in range(5, 8):
        space = charset_size ** length
        secs = space / rate_per_sec
        if   secs < 300:    human = f"{secs:.0f} s  (feasible)"
        elif secs < 86400:  human = f"{secs/3600:.1f} hr (painful)"
        else:               human = f"{secs/86400:.1f} days (no)"
        print(f"  len {length}: {space:>18,} combos -> {human}")

print("Lowercase letters + digits (36):")
keyspace_report(36)
print("\nMixed case + digits (62):")
keyspace_report(62)
# Takeaway: beyond ~5-6 chars, exhaustive search collapses. We must get SMART.


Lowercase letters + digits (36):
charset=36, assuming 5,000,000/s on a CPU

  len 5:         60,466,176 combos -> 12 s  (feasible)
  len 6:      2,176,782,336 combos -> 0.1 hr (painful)
  len 7:     78,364,164,096 combos -> 4.4 hr (painful)

Mixed case + digits (62):
charset=62, assuming 5,000,000/s on a CPU

  len 5:        916,132,832 combos -> 183 s  (feasible)
  len 6:     56,800,235,584 combos -> 3.2 hr (painful)
  len 7:  3,521,614,606,208 combos -> 8.2 days (no)


## Dictionary attack: humans pick words

In [10]:
# Load the REAL rockyou wordlist from your local SecLists clone.
# Jupyter runs locally, so we read the file straight off disk — no upload needed.
import os

ROCKYOU = os.path.expanduser("~/Downloads/SecLists/Passwords/Leaked-Databases/rockyou.txt")
# For a fast first test, swap rockyou.txt -> rockyou-75.txt (a tiny sample) on the line above.

# Sanity check the path before we commit to loading 14M lines.
assert os.path.exists(ROCKYOU), f"File not found: {ROCKYOU}"

# Filter to the password policy (5–7 chars, alphanumeric) AS we load, so we never
# waste time hashing candidates the policy already rules out.
with open(ROCKYOU, encoding="latin-1") as f:
    wordlist = [w.strip() for w in f if 5 <= len(w.strip()) <= 7 and w.strip().isalnum()]

print(f"loaded {len(wordlist):,} candidates from rockyou after the 5–7 alnum filter")

def dictionary_attack(words, targets):
    want = {d["hash"]: u for u, d in targets.items()}
    found = {}
    for w in words:
        hx = commonhash(SALT, w)
        if hx in want:
            found[want[hx]] = w
            print(f"  [+] {want[hx]} -> {w}")
    return found

print("Cracked:", dictionary_attack(wordlist, post_2010))

loaded 4,507,500 candidates from rockyou after the 5–7 alnum filter
  [+] superman -> qwerty
  [+] Mark123 -> Mark123
Cracked: {'superman': 'qwerty', 'Mark123': 'Mark123'}


## Dictionary Attack + Rules

In [11]:
LEET = str.maketrans({"a":"4","e":"3","i":"1","o":"0","s":"5"})

def variants(word):
    """Case + leetspeak transforms of one base word."""
    return {word, word.capitalize(), word.upper(), word.translate(LEET)}

SUFFIXES = ["", "1", "12", "123", "2010", "2011", "01", "11", "99"]

def dictionary_rules_attack(words, targets):
    want = {d["hash"]: u for u, d in targets.items()}
    found = {}
    for w in words:
        for v in variants(w):
            for s in SUFFIXES:
                cand = v + s
                if not (5 <= len(cand) <= 7):      # enforce the policy length
                    continue
                hx = commonhash(SALT, cand)
                if hx in want:
                    found[want[hx]] = cand
                    print(f"  [+] {want[hx]} -> {cand}   (base={w!r})")
    return found

print("Cracked:", dictionary_rules_attack(wordlist, post_2010))


  [+] superman -> qwerty   (base='qwerty')
  [+] Mark123 -> Mark123   (base='mark123')
  [+] Mark123 -> Mark123   (base='MARK123')
  [+] Mark123 -> Mark123   (base='Mark123')
Cracked: {'superman': 'qwerty', 'Mark123': 'Mark123'}


## Stage 5 — Brute force (last resort) with two real optimizations

When smart attacks miss, you exhaust a keyspace — but only a feasible one.
Two optimizations worth knowing:

1. **hashlib copy trick**: the salt prefix is identical every time, so hash it
   once and `.copy()` the state per guess instead of re-hashing the salt.
2. **early structure limits**: only brute-force the length/charset you must.

This cell brute-forces the 5-char space (fast). Try changing `length` to 6 and
watch the time explode — that's the wall from Pivot 1, felt directly.

In [12]:
def brute_force(length, charset, targets, report_every=0):
    want = {d["hash"]: u for u, d in targets.items()}
    base = hashlib.sha1(SALT.encode())   # salt hashed ONCE
    found = {}
    t = time.time()
    for i, combo in enumerate(itertools.product(charset, repeat=length)):
        h = base.copy()                  # reuse salt state
        h.update("".join(combo).encode())
        hx = h.hexdigest()
        if hx in want:
            found[want[hx]] = "".join(combo)
            print(f"  [+] {want[hx]} -> {''.join(combo)}")
    print(f"  ({length}-char sweep done in {time.time()-t:.0f}s)")
    return found

LOWER_ALNUM = "abcdefghijklmnopqrstuvwxyz0123456789"   # 36 chars
# Start with length 5 (fast). Change to 6 to feel the wall (minutes, not seconds).
print("Cracked:", brute_force(6, LOWER_ALNUM, post_2010))


  [+] superman -> qwerty
  (6-char sweep done in 1079s)
Cracked: {'superman': 'qwerty'}


## Results & the real lesson

Running all stages recovers **4 of 6**:

| Account | Password | Fell to | Why it was weak |
|---|---|---|---|
| Tomtom | `54321` | mask | short, digits only |
| security | `121298` | mask | short, digits only |
| superman | `qwerty` | dictionary | famous common password |
| Mark123 | `Mark123` | dictionary+rules | username as password |
| Sparky (Admin) | *uncracked* | — | mixed-case / 7-char, no dictionary word |
| JillC | *uncracked* | — | mixed-case / 7-char, no dictionary word |

**The two that held out followed good practice** — that contrast is the whole
point. Weak passwords fall in seconds; length + mixed case + no dictionary word
beats a CPU attacker.
